In [1]:
# ============================================================
# GENOMIQ AI
# Intelligent Genomic Variant Intelligence Platform
# Google Colab - Complete One-Cell Project
# ============================================================

# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

!pip -q install pandas numpy scikit-learn xgboost shap matplotlib plotly gradio openpyxl

# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import gradio as gr
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ============================================================
# 3. PROJECT DIRECTORIES
# ============================================================

BASE_DIR = "/content/GENOMIQ-AI"
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 4. GLOBAL SETTINGS
# ============================================================

np.random.seed(42)

FEATURES = [
    "PrimateAI-3D",
    "SpliceAI",
    "PhyloP"
]

ENGINEERED_FEATURES = [
    "PrimateAI-3D",
    "SpliceAI",
    "PhyloP",
    "AI_Combined_Score",
    "Conservation_Risk",
    "Splicing_Risk"
]

GLOBAL_DF = None
LATEST_MODELS = {}

# ============================================================
# 5. GENERATE SYNTHETIC GENOMIC DATA
# ============================================================

def generate_demo_dataset(n=1200):

    genes = [
        "BRCA1",
        "BRCA2",
        "TP53",
        "EGFR",
        "KRAS",
        "APC",
        "PTEN",
        "CFTR",
        "MLH1",
        "MSH2",
        "PIK3CA",
        "VHL",
        "RB1",
        "ATM",
        "PALB2"
    ]

    chromosomes = np.random.choice(
        [
            "1", "2", "3", "4", "5", "6",
            "7", "8", "9", "10", "11",
            "12", "13", "14", "15", "16",
            "17", "18", "19", "20", "21",
            "22", "X"
        ],
        size=n
    )

    amino_acids = list("ACDEFGHIKLMNPQRSTVWY")

    hgvsp = [
        f"p.{np.random.choice(amino_acids)}"
        f"{np.random.randint(1, 700)}"
        f"{np.random.choice(amino_acids)}"
        for _ in range(n)
    ]

    data = pd.DataFrame({
        "Chromosome": chromosomes,
        "Position": np.random.randint(
            10000,
            250000000,
            size=n
        ),
        "Gene": np.random.choice(
            genes,
            size=n
        ),
        "REF": np.random.choice(
            ["A", "C", "G", "T"],
            size=n
        ),
        "ALT": np.random.choice(
            ["A", "C", "G", "T"],
            size=n
        ),
        "HGVSP": hgvsp,
        "PrimateAI-3D": np.clip(
            np.random.beta(2.2, 2.0, n),
            0,
            1
        ),
        "SpliceAI": np.clip(
            np.random.beta(1.8, 3.0, n),
            0,
            1
        ),
        "PhyloP": np.random.normal(
            2.2,
            1.8,
            n
        )
    })

    risk_score = (
        2.2 * data["PrimateAI-3D"]
        + 1.8 * data["SpliceAI"]
        + 0.35 * data["PhyloP"]
        + np.random.normal(0, 0.7, n)
    )

    threshold = np.median(risk_score)

    data["Target"] = (
        risk_score > threshold
    ).astype(int)

    for column in FEATURES:

        missing_index = np.random.choice(
            data.index,
            size=int(n * 0.04),
            replace=False
        )

        data.loc[
            missing_index,
            column
        ] = np.nan

    return data


GLOBAL_DF = generate_demo_dataset()

DEMO_PATH = os.path.join(
    DATA_DIR,
    "demo_variants.csv"
)

GLOBAL_DF.to_csv(
    DEMO_PATH,
    index=False
)

# ============================================================
# 6. DATASET FUNCTIONS
# ============================================================

def normalize_uploaded_file(file):

    if file is None:
        return None

    if isinstance(file, str):
        return file

    if hasattr(file, "name"):
        return file.name

    return str(file)


def load_dataset(file):

    global GLOBAL_DF

    if file is None:

        return GLOBAL_DF.copy()

    path = normalize_uploaded_file(file)

    try:

        if path.lower().endswith(".tsv"):

            data = pd.read_csv(
                path,
                sep="\t"
            )

        else:

            data = pd.read_csv(
                path
            )

        GLOBAL_DF = data.copy()

        return GLOBAL_DF

    except Exception as error:

        return pd.DataFrame({
            "Error": [str(error)]
        })


def dataset_audit(data):

    if data is None or len(data) == 0:

        return pd.DataFrame({
            "Message": [
                "No dataset available"
            ]
        })

    return pd.DataFrame({

        "Column": data.columns,

        "Data Type": [
            str(data[column].dtype)
            for column in data.columns
        ],

        "Missing Values": [
            data[column].isna().sum()
            for column in data.columns
        ],

        "Missing Percentage": [
            round(
                data[column].isna().mean() * 100,
                2
            )
            for column in data.columns
        ],

        "Unique Values": [
            data[column].nunique()
            for column in data.columns
        ]
    })


def dataset_statistics(data):

    if data is None or len(data) == 0:
        return pd.DataFrame()

    result = (
        data.describe(
            include="all"
        )
        .T
        .reset_index()
        .rename(
            columns={
                "index": "Feature"
            }
        )
    )

    return result


def dataset_summary(data):

    if data is None or len(data) == 0:

        return """
No dataset loaded.
"""

    numeric_columns = data.select_dtypes(
        include=np.number
    ).columns

    summary = f"""
Dataset Summary

Rows: {len(data)}
Columns: {len(data.columns)}
Numeric Columns: {len(numeric_columns)}
Missing Cells: {int(data.isna().sum().sum())}
Duplicate Rows: {int(data.duplicated().sum())}
"""

    if "Gene" in data.columns:

        summary += (
            f"\nUnique Genes: "
            f"{data['Gene'].nunique()}"
        )

    return summary


def process_dataset(file):

    data = load_dataset(file)

    audit = dataset_audit(data)

    statistics = dataset_statistics(data)

    summary = dataset_summary(data)

    preview = data.head(30)

    return (
        preview,
        audit,
        statistics,
        summary
    )

# ============================================================
# 7. FEATURE ENGINEERING
# ============================================================

def engineer_features(data):

    d = data.copy()

    for feature in FEATURES:

        if feature not in d.columns:

            d[feature] = 0.0

    for feature in FEATURES:

        d[feature] = pd.to_numeric(
            d[feature],
            errors="coerce"
        )

    primate = d[
        "PrimateAI-3D"
    ].fillna(
        d["PrimateAI-3D"].median()
    )

    splice = d[
        "SpliceAI"
    ].fillna(
        d["SpliceAI"].median()
    )

    phylo = d[
        "PhyloP"
    ].fillna(
        d["PhyloP"].median()
    )

    max_abs_phylo = (
        abs(phylo).max()
        + 1e-9
    )

    d[
        "AI_Combined_Score"
    ] = (
        0.5 * primate
        + 0.3 * splice
        + 0.2 * (
            phylo / max_abs_phylo
        )
    )

    d[
        "Conservation_Risk"
    ] = (
        phylo > 2.5
    ).astype(int)

    d[
        "Splicing_Risk"
    ] = (
        splice > 0.5
    ).astype(int)

    return d

# ============================================================
# 8. TARGET HANDLING
# ============================================================

def get_target(data):

    if "Target" in data.columns:

        target = pd.to_numeric(
            data["Target"],
            errors="coerce"
        )

    elif "target" in data.columns:

        target = pd.to_numeric(
            data["target"],
            errors="coerce"
        )

    elif "Pathogenic" in data.columns:

        target = pd.to_numeric(
            data["Pathogenic"],
            errors="coerce"
        )

    else:

        engineered = engineer_features(
            data
        )

        score = (
            2.0 * engineered[
                "PrimateAI-3D"
            ].fillna(
                engineered[
                    "PrimateAI-3D"
                ].median()
            )
            +
            1.7 * engineered[
                "SpliceAI"
            ].fillna(
                engineered[
                    "SpliceAI"
                ].median()
            )
            +
            0.35 * engineered[
                "PhyloP"
            ].fillna(
                engineered[
                    "PhyloP"
                ].median()
            )
        )

        target = (
            score > score.median()
        ).astype(int)

    return target


def get_training_data(data):

    processed = engineer_features(
        data
    )

    for feature in ENGINEERED_FEATURES:

        if feature not in processed.columns:

            processed[feature] = 0

    X = processed[
        ENGINEERED_FEATURES
    ].copy()

    y = get_target(
        processed
    )

    valid = y.notna()

    X = X.loc[
        valid
    ].reset_index(
        drop=True
    )

    y = y.loc[
        valid
    ].astype(int).reset_index(
        drop=True
    )

    processed = processed.reset_index(
        drop=True
    )

    return (
        processed,
        X,
        y,
        ENGINEERED_FEATURES
    )

# ============================================================
# 9. MODEL TRAINING
# ============================================================

def create_models():

    models = {

        "Logistic Regression":
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "scaler",
                StandardScaler()
            ),

            (
                "model",
                LogisticRegression(
                    max_iter=3000
                )
            )
        ]),

        "Random Forest":
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]),

        "XGBoost":
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "model",
                XGBClassifier(
                    n_estimators=250,
                    max_depth=5,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    eval_metric="logloss",
                    random_state=42
                )
            )
        ]),

        "Support Vector Machine":
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "scaler",
                StandardScaler()
            ),

            (
                "model",
                SVC(
                    probability=True,
                    class_weight="balanced",
                    random_state=42
                )
            )
        ])
    }

    return models


def train_models(data):

    (
        processed,
        X,
        y,
        feature_columns
    ) = get_training_data(
        data
    )

    if len(np.unique(y)) < 2:

        raise ValueError(
            "The target must contain at least two classes."
        )

    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,
            test_size=0.20,
            random_state=42,
            stratify=y
        )
    )

    models = create_models()

    results = []

    trained_models = {}

    for name, model in models.items():

        model.fit(
            X_train,
            y_train
        )

        predictions = model.predict(
            X_test
        )

        probabilities = (
            model.predict_proba(
                X_test
            )[:, 1]
        )

        results.append({

            "Model": name,

            "Accuracy": round(
                accuracy_score(
                    y_test,
                    predictions
                ),
                4
            ),

            "Precision": round(
                precision_score(
                    y_test,
                    predictions,
                    zero_division=0
                ),
                4
            ),

            "Recall": round(
                recall_score(
                    y_test,
                    predictions,
                    zero_division=0
                ),
                4
            ),

            "F1": round(
                f1_score(
                    y_test,
                    predictions,
                    zero_division=0
                ),
                4
            ),

            "ROC-AUC": round(
                roc_auc_score(
                    y_test,
                    probabilities
                ),
                4
            )
        })

        trained_models[name] = model

    results_df = (
        pd.DataFrame(results)
        .sort_values(
            "ROC-AUC",
            ascending=False
        )
        .reset_index(drop=True)
    )

    best_name = results_df.iloc[
        0
    ]["Model"]

    best_model = trained_models[
        best_name
    ]

    results_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "model_comparison.csv"
        ),
        index=False
    )

    return (
        results_df,
        trained_models,
        best_name,
        best_model,
        X_train,
        X_test,
        y_train,
        y_test,
        feature_columns,
        processed
    )


def train_and_store(data):

    global LATEST_MODELS

    (
        results,
        trained_models,
        best_name,
        best_model,
        X_train,
        X_test,
        y_train,
        y_test,
        feature_columns,
        processed
    ) = train_models(
        data
    )

    LATEST_MODELS = {

        "models":
        trained_models,

        "best_name":
        best_name,

        "best_model":
        best_model,

        "feature_columns":
        feature_columns,

        "processed":
        processed,

        "X_train":
        X_train,

        "X_test":
        X_test,

        "y_train":
        y_train,

        "y_test":
        y_test
    }

    return (
        results,
        best_name
    )

# ============================================================
# 10. TRAINING INTERFACE
# ============================================================

def train_interface():

    try:

        results, best_name = train_and_store(
            GLOBAL_DF
        )

        chart = px.bar(
            results,
            x="Model",
            y="ROC-AUC",
            text_auto=True,
            title="Model ROC-AUC Comparison"
        )

        status = (
            "Training completed successfully.\n"
            f"Selected model: {best_name}"
        )

        return (
            results,
            chart,
            status
        )

    except Exception as error:

        return (
            pd.DataFrame({
                "Error": [str(error)]
            }),
            None,
            f"Training failed: {error}"
        )

# ============================================================
# 11. BULK PREDICTION
# ============================================================

def bulk_prediction():

    global LATEST_MODELS

    try:

        if not LATEST_MODELS:

            train_and_store(
                GLOBAL_DF
            )

        processed = LATEST_MODELS[
            "processed"
        ].copy()

        model = LATEST_MODELS[
            "best_model"
        ]

        feature_columns = LATEST_MODELS[
            "feature_columns"
        ]

        X = processed[
            feature_columns
        ]

        predictions = model.predict(
            X
        )

        probabilities = (
            model.predict_proba(
                X
            )[:, 1]
        )

        result = processed.copy()

        result[
            "Prediction"
        ] = predictions

        result[
            "Pathogenicity_Probability"
        ] = np.round(
            probabilities,
            4
        )

        result[
            "Risk_Category"
        ] = pd.cut(
            probabilities,
            bins=[
                -np.inf,
                0.33,
                0.66,
                np.inf
            ],
            labels=[
                "Lower",
                "Moderate",
                "Higher"
            ]
        )

        result[
            "Model_Used"
        ] = LATEST_MODELS[
            "best_name"
        ]

        output_path = os.path.join(
            OUTPUT_DIR,
            "genomiq_predictions.csv"
        )

        result.to_csv(
            output_path,
            index=False
        )

        return (
            result.head(100),
            output_path
        )

    except Exception as error:

        return (
            pd.DataFrame({
                "Error": [str(error)]
            }),
            None
        )

# ============================================================
# 12. INDIVIDUAL VARIANT PREDICTION
# ============================================================

def predict_single_variant(
    primateai,
    spliceai,
    phylop
):

    global LATEST_MODELS

    try:

        if not LATEST_MODELS:

            train_and_store(
                GLOBAL_DF
            )

        model = LATEST_MODELS[
            "best_model"
        ]

        sample = pd.DataFrame({

            "PrimateAI-3D":
            [primateai],

            "SpliceAI":
            [spliceai],

            "PhyloP":
            [phylop]
        })

        sample = engineer_features(
            sample
        )

        feature_columns = LATEST_MODELS[
            "feature_columns"
        ]

        X = sample[
            feature_columns
        ]

        prediction = int(
            model.predict(X)[0]
        )

        probability = float(
            model.predict_proba(X)[0, 1]
        )

        if probability < 0.33:

            category = "Lower"

        elif probability < 0.66:

            category = "Moderate"

        else:

            category = "Higher"

        if prediction == 1:

            interpretation = (
                "Potentially Pathogenic"
            )

        else:

            interpretation = (
                "Potentially Non-Pathogenic"
            )

        result = f"""
GENOMIQ AI VARIANT ANALYSIS

Prediction:
{interpretation}

Pathogenicity Probability:
{probability:.2%}

Risk Category:
{category}

Model:
{LATEST_MODELS["best_name"]}

Input Features:
PrimateAI-3D: {primateai}
SpliceAI: {spliceai}
PhyloP: {phylop}

This output is a machine-learning research demonstration
and should not be used as a clinical diagnosis.
"""

        return result

    except Exception as error:

        return f"Prediction failed: {error}"

# ============================================================
# 13. PCA AND K-MEANS
# ============================================================

def run_clustering():

    try:

        data = engineer_features(
            GLOBAL_DF
        )

        clustering_features = [
            "PrimateAI-3D",
            "SpliceAI",
            "PhyloP"
        ]

        X = data[
            clustering_features
        ].copy()

        imputer = SimpleImputer(
            strategy="median"
        )

        X = imputer.fit_transform(
            X
        )

        scaler = StandardScaler()

        X_scaled = scaler.fit_transform(
            X
        )

        kmeans = KMeans(
            n_clusters=4,
            n_init=20,
            random_state=42
        )

        cluster_labels = (
            kmeans.fit_predict(
                X_scaled
            )
        )

        pca = PCA(
            n_components=2,
            random_state=42
        )

        components = pca.fit_transform(
            X_scaled
        )

        plot_data = pd.DataFrame({

            "PCA1":
            components[:, 0],

            "PCA2":
            components[:, 1],

            "Cluster":
            cluster_labels.astype(str)
        })

        plot = px.scatter(
            plot_data,
            x="PCA1",
            y="PCA2",
            color="Cluster",
            title="PCA and K-Means Variant Clusters"
        )

        return (
            plot_data.head(100),
            plot
        )

    except Exception as error:

        return (
            pd.DataFrame({
                "Error": [str(error)]
            }),
            None
        )

# ============================================================
# 14. ANOMALY DETECTION
# ============================================================

def run_anomaly_detection():

    try:

        data = engineer_features(
            GLOBAL_DF
        )

        anomaly_features = [
            "PrimateAI-3D",
            "SpliceAI",
            "PhyloP"
        ]

        X = data[
            anomaly_features
        ]

        imputer = SimpleImputer(
            strategy="median"
        )

        X = imputer.fit_transform(
            X
        )

        scaler = StandardScaler()

        X_scaled = scaler.fit_transform(
            X
        )

        detector = IsolationForest(
            contamination=0.05,
            random_state=42
        )

        labels = detector.fit_predict(
            X_scaled
        )

        scores = detector.decision_function(
            X_scaled
        )

        result = data.copy()

        result[
            "Anomaly"
        ] = np.where(
            labels == -1,
            "Anomalous",
            "Normal"
        )

        result[
            "Anomaly_Score"
        ] = np.round(
            scores,
            5
        )

        return result.head(100)

    except Exception as error:

        return pd.DataFrame({
            "Error": [str(error)]
        })

# ============================================================
# 15. SHAP EXPLAINABILITY
# ============================================================

def run_shap():

    global LATEST_MODELS

    try:

        if not LATEST_MODELS:

            train_and_store(
                GLOBAL_DF
            )

        model = LATEST_MODELS[
            "best_model"
        ]

        processed = LATEST_MODELS[
            "processed"
        ]

        feature_columns = LATEST_MODELS[
            "feature_columns"
        ]

        X = processed[
            feature_columns
        ].copy()

        pipeline_imputer = (
            model.named_steps["imputer"]
        )

        X_imputed = pipeline_imputer.transform(
            X
        )

        base_model = model.named_steps[
            "model"
        ]

        # Tree-based explainability

        if isinstance(
            base_model,
            (
                RandomForestClassifier,
                XGBClassifier
            )
        ):

            explainer = shap.TreeExplainer(
                base_model
            )

            shap_values = (
                explainer.shap_values(
                    X_imputed
                )
            )

            if isinstance(
                shap_values,
                list
            ):

                shap_values = shap_values[1]

            plt.figure(
                figsize=(10, 6)
            )

            shap.summary_plot(
                shap_values,
                X_imputed,
                feature_names=feature_columns,
                show=False
            )

            plt.tight_layout()

            output_path = os.path.join(
                OUTPUT_DIR,
                "shap_summary.png"
            )

            plt.savefig(
                output_path,
                dpi=160,
                bbox_inches="tight"
            )

            plt.close()

            return output_path

        # Linear or SVM explainability

        if isinstance(
            base_model,
            LogisticRegression
        ):

            scaler = model.named_steps[
                "scaler"
            ]

            X_scaled = scaler.transform(
                X_imputed
            )

            explainer = shap.Explainer(
                base_model,
                X_scaled
            )

            shap_values = explainer(
                X_scaled[:200]
            )

            plt.figure(
                figsize=(10, 6)
            )

            shap.plots.beeswarm(
                shap_values,
                show=False
            )

            plt.tight_layout()

            output_path = os.path.join(
                OUTPUT_DIR,
                "shap_summary.png"
            )

            plt.savefig(
                output_path,
                dpi=160,
                bbox_inches="tight"
            )

            plt.close()

            return output_path

        if isinstance(
            base_model,
            SVC
        ):

            return create_fallback_importance(
                base_model,
                feature_columns
            )

        return create_fallback_importance(
            base_model,
            feature_columns
        )

    except Exception as error:

        return create_fallback_importance(
            None,
            LATEST_MODELS.get(
                "feature_columns",
                ENGINEERED_FEATURES
            ),
            error
        )


def create_fallback_importance(
    model,
    feature_columns,
    error=None
):

    if model is not None:

        if hasattr(
            model,
            "feature_importances_"
        ):

            importance = (
                model.feature_importances_
            )

        elif hasattr(
            model,
            "coef_"
        ):

            importance = np.abs(
                model.coef_[0]
            )

        else:

            importance = np.ones(
                len(feature_columns)
            )

    else:

        importance = np.ones(
            len(feature_columns)
        )

    importance_df = pd.DataFrame({

        "Feature":
        feature_columns,

        "Importance":
        importance
    })

    importance_df = (
        importance_df
        .sort_values(
            "Importance",
            ascending=False
        )
    )

    figure = px.bar(
        importance_df,
        x="Importance",
        y="Feature",
        orientation="h",
        title="Feature Importance"
    )

    output_path = os.path.join(
        OUTPUT_DIR,
        "feature_importance.html"
    )

    figure.write_html(
        output_path
    )

    return output_path

# ============================================================
# 16. INITIAL MODEL TRAINING
# ============================================================

try:

    train_and_store(
        GLOBAL_DF
    )

except Exception as error:

    print(
        f"Initial model training failed: {error}"
    )

# ============================================================
# 17. CUSTOM INTERFACE
# ============================================================

custom_css = """
body {
    background: #f7f8fa;
}

.gradio-container {
    max-width: 1400px !important;
}

.title {
    text-align: center;
    font-size: 32px;
    font-weight: 700;
}

.subtitle {
    text-align: center;
    color: #555;
    margin-bottom: 20px;
}

.notice {
    padding: 12px;
    border-radius: 8px;
    background: #f2f2f2;
    margin-bottom: 15px;
}
"""

# ============================================================
# 18. BUILD GRADIO APPLICATION
# ============================================================

with gr.Blocks(
    title="GENOMIQ AI",
    css=custom_css
) as app:

    gr.Markdown(
        """
        <div class="title">
        GENOMIQ AI
        </div>
        """,
        elem_classes="title"
    )

    gr.Markdown(
        """
        <div class="subtitle">
        Intelligent Genomic Variant Intelligence Platform
        </div>
        """
    )

    gr.Markdown(
        """
        Machine Learning, genomic feature analysis,
        explainability, clustering and anomaly detection.
        """
    )

    gr.Markdown(
        """
        Research and education prototype. Outputs should not
        be interpreted as clinical diagnoses.
        """,
        elem_classes="notice"
    )

    # ========================================================
    # DATASET TAB
    # ========================================================

    with gr.Tab("Dataset"):

        gr.Markdown(
            "Upload a CSV or TSV file or use the included synthetic dataset."
        )

        file_input = gr.File(
            label="Upload Dataset",
            file_types=[
                ".csv",
                ".tsv"
            ],
            type="filepath"
        )

        load_button = gr.Button(
            "Load Dataset",
            variant="primary"
        )

        dataset_preview = gr.Dataframe(
            value=GLOBAL_DF.head(20),
            label="Dataset Preview"
        )

        dataset_audit_output = gr.Dataframe(
            label="Dataset Audit"
        )

        dataset_stats_output = gr.Dataframe(
            label="Dataset Statistics"
        )

        dataset_summary_output = gr.Textbox(
            label="Dataset Summary",
            lines=8
        )

        load_button.click(
            process_dataset,
            inputs=file_input,
            outputs=[
                dataset_preview,
                dataset_audit_output,
                dataset_stats_output,
                dataset_summary_output
            ]
        )

    # ========================================================
    # MODEL TAB
    # ========================================================

    with gr.Tab("Model Training"):

        gr.Markdown(
            """
            Available models:

            Logistic Regression

            Random Forest

            XGBoost

            Support Vector Machine
            """
        )

        train_button = gr.Button(
            "Train and Compare Models",
            variant="primary"
        )

        model_results = gr.Dataframe(
            label="Model Performance"
        )

        model_chart = gr.Plot(
            label="ROC-AUC Comparison"
        )

        training_status = gr.Textbox(
            label="Training Status",
            lines=4
        )

        train_button.click(
            train_interface,
            outputs=[
                model_results,
                model_chart,
                training_status
            ]
        )

    # ========================================================
    # SINGLE VARIANT TAB
    # ========================================================

    with gr.Tab("Variant Analysis"):

        gr.Markdown(
            """
            Enter genomic prediction features to obtain
            a model-generated research output.
            """
        )

        with gr.Row():

            primate_input = gr.Number(
                value=0.55,
                minimum=0,
                maximum=1,
                label="PrimateAI-3D"
            )

            splice_input = gr.Number(
                value=0.40,
                minimum=0,
                maximum=1,
                label="SpliceAI"
            )

            phylo_input = gr.Number(
                value=3.0,
                label="PhyloP"
            )

        variant_button = gr.Button(
            "Analyze Variant",
            variant="primary"
        )

        variant_output = gr.Textbox(
            label="Variant Analysis",
            lines=20
        )

        variant_button.click(
            predict_single_variant,
            inputs=[
                primate_input,
                splice_input,
                phylo_input
            ],
            outputs=variant_output
        )

    # ========================================================
    # BULK PREDICTION TAB
    # ========================================================

    with gr.Tab("Bulk Predictions"):

        gr.Markdown(
            """
            Generate predictions for the currently loaded dataset.
            """
        )

        bulk_button = gr.Button(
            "Generate Predictions",
            variant="primary"
        )

        bulk_output = gr.Dataframe(
            label="Prediction Results"
        )

        prediction_download = gr.File(
            label="Download Predictions"
        )

        bulk_button.click(
            bulk_prediction,
            outputs=[
                bulk_output,
                prediction_download
            ]
        )

    # ========================================================
    # CLUSTERING TAB
    # ========================================================

    with gr.Tab("PCA and K-Means"):

        gr.Markdown(
            """
            PCA reduces the feature space to two dimensions,
            while K-Means groups variants into clusters.
            """
        )

        cluster_button = gr.Button(
            "Run Clustering",
            variant="primary"
        )

        cluster_table = gr.Dataframe(
            label="Cluster Results"
        )

        cluster_plot = gr.Plot(
            label="PCA Cluster Visualization"
        )

        cluster_button.click(
            run_clustering,
            outputs=[
                cluster_table,
                cluster_plot
            ]
        )

    # ========================================================
    # ANOMALY TAB
    # ========================================================

    with gr.Tab("Anomaly Detection"):

        gr.Markdown(
            """
            Isolation Forest is used to identify unusual
            combinations of genomic feature values.
            """
        )

        anomaly_button = gr.Button(
            "Detect Anomalies",
            variant="primary"
        )

        anomaly_output = gr.Dataframe(
            label="Anomaly Results"
        )

        anomaly_button.click(
            run_anomaly_detection,
            outputs=anomaly_output
        )

    # ========================================================
    # EXPLAINABILITY TAB
    # ========================================================

    with gr.Tab("Explainable AI"):

        gr.Markdown(
            """
            SHAP-based explainability is generated for
            supported tree and linear models.
            """
        )

        shap_button = gr.Button(
            "Generate Explainability",
            variant="primary"
        )

        shap_output = gr.File(
            label="Explainability Output"
        )

        shap_button.click(
            run_shap,
            outputs=shap_output
        )

    # ========================================================
    # ABOUT TAB
    # ========================================================

    with gr.Tab("About"):

        gr.Markdown(
            """
            ## GENOMIQ AI

            GENOMIQ AI is a research-oriented genomic
            machine-learning platform.

            ### Processing Pipeline

            Genomic Dataset

            ->

            Data Validation

            ->

            Feature Engineering

            ->

            Model Training

            ->

            Model Comparison

            ->

            Prediction

            ->

            Explainability

            ->

            PCA and K-Means

            ->

            Anomaly Detection

            ### Technology Stack

            Python

            Pandas

            NumPy

            Scikit-learn

            XGBoost

            SHAP

            Plotly

            Gradio

            Google Colab

            ### Input Features

            PrimateAI-3D

            SpliceAI

            PhyloP

            ### Output

            Prediction

            Pathogenicity probability

            Risk category

            Model comparison

            Cluster assignments

            Anomaly scores

            Explainability output

            ### Important Note

            The included dataset is synthetic and is intended
            for demonstration, experimentation and portfolio use.

            The system is not a clinical diagnostic tool.
            """
        )

# ============================================================
# 19. LAUNCH APPLICATION
# ============================================================

print("=" * 70)
print("GENOMIQ AI")
print("=" * 70)
print("Application is ready.")
print(f"Demo dataset: {DEMO_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print("=" * 70)

app.launch(
    share=True,
    debug=False
)

GENOMIQ AI
Application is ready.
Demo dataset: /content/GENOMIQ-AI/data/demo_variants.csv
Output directory: /content/GENOMIQ-AI/outputs
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fdcfb08e46ce3ed0fa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
